# TV3 Offline Evaluation — Synthetic Example

Notebook này minh họa cách gọi evaluator bằng **dữ liệu tổng hợp được khai báo ngay trong notebook**. Nó không đọc dataset, JSONL, CSV, metadata hoặc human annotations từ repository; không gọi LLM, image backend, GPU hay tạo human labels.

Khi đánh giá dữ liệu thực, hãy chuẩn bị input và inventory đã được phê duyệt ở vị trí riêng, theo quy trình nội bộ. Không commit các tệp đó vào public repository.


## 1. Import evaluator

Chạy notebook từ repository root để Python tìm thấy package `src`.

In [ ]:
from src.evaluation.evaluate import evaluate_records
from pprint import pprint


## 2. Tạo storyboard tổng hợp

Storyboard dưới đây chỉ phục vụ kiểm thử cấu trúc evaluator; nội dung và nhãn sự kiện đều là giả lập.

In [ ]:
import json

synthetic_storyboard = {
    "summary": "Một cậu bé tìm thấy chiếc chìa khóa.",
    "characters": [
        {"id": "boy", "visual": "A young boy with a blue shirt."}
    ],
    "scenes": [
        {
            "index": index,
            "caption_vi": caption,
            "image_prompt_en": f"Synthetic comic scene {index}",
            "characters": ["boy"],
        }
        for index, caption in enumerate(
            [
                "Cậu bé đi vào khu rừng.",
                "Cậu bé nhìn thấy chiếc chìa khóa.",
                "Cậu bé nhặt chiếc chìa khóa lên.",
                "Cậu bé mở chiếc rương bằng chìa khóa.",
            ],
            start=1,
        )
    ],
}

synthetic_raw_output = json.dumps(synthetic_storyboard, ensure_ascii=False)


## 3. Tạo annotation và event inventory tổng hợp

Event IDs trong annotation phải khớp chính xác inventory. Scene index được dùng để minh họa Event Order.

In [ ]:
case_id = "synthetic_case_001"
synthetic_event_ids = [
    f"{case_id}_event_01",
    f"{case_id}_event_02",
]

synthetic_records = [
    {
        "case_id": case_id,
        "raw_output": synthetic_raw_output,
        "keyword_groups": [["cậu bé"], ["chiếc chìa khóa"]],
        "event_annotations": [
            {
                "event_id": synthetic_event_ids[0],
                "reference_order": 1,
                "annotator_1_label": "covered",
                "annotator_1_scene_index": 1,
                "annotator_2_label": "covered",
                "annotator_2_scene_index": 1,
                "adjudicated_label": "covered",
                "adjudicated_scene_index": 1,
            },
            {
                "event_id": synthetic_event_ids[1],
                "reference_order": 2,
                "annotator_1_label": "covered",
                "annotator_1_scene_index": 3,
                "annotator_2_label": "covered",
                "annotator_2_scene_index": 3,
                "adjudicated_label": "covered",
                "adjudicated_scene_index": 3,
            },
        ],
    }
]

synthetic_inventory = {case_id: synthetic_event_ids}


## 4. Chạy evaluator

Inventory gate ở đây chỉ kiểm tra tính khớp case/event giữa hai fixture tổng hợp. Nó không chứng minh tính đúng đắn ngữ nghĩa của sự kiện hoặc annotation.

In [ ]:
report = evaluate_records(
    synthetic_records,
    event_inventory=synthetic_inventory,
)

print("Planned cases:", report["planned_cases"])
print("Event Coverage status:", report["event_coverage_status"])
print("Event Order status:", report["event_order_status"])
print("Inventory status:", report["event_inventory_status"])
pprint(report["metrics"])


## 5. Diễn giải kết quả

- Four Scene Rate và Schema Valid Rate sử dụng planned cases làm mẫu số.
- Event Coverage sử dụng các reference events có annotation; `uncertain` không được tính vào tử số.
- Event Order Accuracy chỉ tính trên các cặp evaluable; Pair Coverage phải được báo riêng.
- Inventory gate xác nhận độ đầy đủ của case/event IDs, không xác nhận semantic correctness.
- Các kết quả từ notebook này là **synthetic smoke-test output**, không phải kết quả benchmark hoặc ablation.